# TN1 phần B — TCN và DS-TCN

Chạy **song song** với `TN1_LSTM.ipynb` ở một phiên Colab khác. Hai notebook độc lập hoàn toàn, không cần chờ nhau.

| notebook | chạy gì | thời gian |
|---|---|---|
| TN1_LSTM.ipynb | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 giờ |
| **TN1_TCN_DSTCN.ipynb** ← đang mở | TCN-64 và DS-TCN-64: 4 fold CV mỗi cái | ~2.2 giờ |
| TN1.ipynb | gộp kết quả, so sánh, chạy GHIJ cho kiến trúc thắng | ~1.5 giờ |

## Hai kiến trúc

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |

Khác nhau **đúng một chỗ** — phép tích chập trong mỗi khối:

```python
# TCN thường
nn.Conv1d(64, 64, kernel_size=3, dilation=d)

# DS-TCN, tách làm hai bước (Howard et al. 2017, mục 3.1)
nn.Conv1d(64, 64, 3, dilation=d, groups=64)   # depthwise: mỗi kênh một bộ lọc
nn.Conv1d(64, 64, 1)                          # pointwise: chỉ trộn kênh
```

Mọi thứ khác giống hệt: `kernel=3`, `n_blocks=6`, hai tầng conv mỗi khối, `dropout=0.0`, BatchNorm, ReLU, nối tắt. Trích dẫn từng tham số ở [`docs/THAM_CHIEU.md`](../docs/THAM_CHIEU.md).

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

Cấu hình giữ nguyên như MobiVital công bố: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN. Một seed.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo tác giả.


In [3]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
!git clone -q --branch submission --single-branch https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 87f8756
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB — chỉ train trên cửa sổ đã cắt, chấm trên `by_user/*.npz`.


In [6]:
!python scripts/restore_processed_data_on_drive.py


by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. TCN-64 — 4 fold CV

Khoảng **1 giờ**.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64


thực nghiệm tn1  -> runs/tn1/
cấu hình tcn_c64_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03438  pearson 0.4631   0.6 phút
epoch  1  mse 0.02013  pearson 0.5452   1.1 phút
epoch  2  mse 0.01884  pearson 0.5694   1.7 phút
epoch  3  mse 0.01795  pearson 0.5822   2.2 phút
epoch  4  mse 0.01734  pearson 0.5904   2.7 phút
epoch  5  mse 0.01677  pearson 0.5989   3.3 phút
epoch  6  mse 0.01640  pearson 0.6046   3.8 phút
epoch  7  mse 0.01600  pearson 0.6108   4.3 phút
epoch  8  mse 0.01567  pearson 0.6154   4.9 phút
epoch  9  mse 0.01537  pearson 0.6196   5.4 phút
epoch 10  mse 0.01515  pearson 0.6235   6.0 phút
epoch 11  mse 0.01484  pearson 0.6271   6.5 phút
epoch 12  mse 0.01459  pearson 0.6300   7.1 phút
epoch 13  mse 0.01434  pearson 0.6323   7.6 phút
epoch 14  mse 0.01410  pearson 0.6348   8.1 phút
epoch 15  mse 0.01389  pearson 0.6373   8.7 phút
epoch 16  mse 0.01365

## 3. DS-TCN-64 — 4 fold CV

Khoảng **1.2 giờ**.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64


thực nghiệm tn1  -> runs/tn1/
cấu hình ds_tcn_c64_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.05256  pearson 0.4147   0.6 phút
epoch  1  mse 0.02089  pearson 0.5300   1.2 phút
epoch  2  mse 0.01917  pearson 0.5617   1.8 phút
epoch  3  mse 0.01828  pearson 0.5788   2.5 phút
epoch  4  mse 0.01759  pearson 0.5897   3.1 phút
epoch  5  mse 0.01710  pearson 0.5973   3.7 phút
epoch  6  mse 0.01675  pearson 0.6037   4.4 phút
epoch  7  mse 0.01641  pearson 0.6088   5.0 phút
epoch  8  mse 0.01614  pearson 0.6127   5.6 phút
epoch  9  mse 0.01592  pearson 0.6169   6.2 phút
epoch 10  mse 0.01567  pearson 0.6204   6.9 phút
epoch 11  mse 0.01549  pearson 0.6227   7.5 phút
epoch 12  mse 0.01528  pearson 0.6255   8.1 phút
epoch 13  mse 0.01509  pearson 0.6275   8.7 phút
epoch 14  mse 0.01493  pearson 0.6290   9.4 phút
epoch 15  mse 0.01478  pearson 0.6310   10.0 phút
epoch 16  mse 0.0

## 3b. TCN và DS-TCN — seed 1 và 2

Ba cấu hình TN1 mới chạy **một seed**. Chênh lệch giữa DS-TCN và TCN là 0.0036,
nhỏ hơn nhiều so với nhiễu hạt giống đo được trên GHIJ (0.0126) — nên chưa
phân định được cấu hình nào hơn.

Chạy thêm seed 1 và 2 để mỗi cấu hình có `cv_mean ± std` thay vì một con số đơn.
Phải chạy cho **cả ba** model: so `mean` của ba seed với một con số đơn là so
hai đại lượng khác nhau.

Giữ nguyên mọi thứ khác — 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0.9,
bốn fold cũ. Chỉ đổi `--seed`.

**Chỉ chạy hai thứ:** các ô setup ở mục 1, rồi ô ngay dưới đây.

Mọi ô khác đã có kết quả seed 0 — bấm lại là mất output cũ và tốn thêm hàng
giờ. Ô nén ở mục cuối cũng không bấm: script đã tự nén sau mỗi fold rồi, và
ô đó ghi đè lên tệp nén của seed 0.

Sau **mỗi fold** script tự nén kết quả rồi chép sang Drive, tên tệp chứa cấu
hình nên hai phiên chạy song song không đè lên nhau. Colab ngắt phiên thì chạy
lại đúng ô này, nó bỏ qua fold đã xong và đi tiếp.

Khoảng 5 giờ.

In [7]:
!python scripts/run_cv.py --experiment tn1 --model tcn    --channels 64 --seed 1
!python scripts/run_cv.py --experiment tn1 --model tcn    --channels 64 --seed 2
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64 --seed 1
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64 --seed 2

thực nghiệm tn1  -> runs/tn1/
cấu hình tcn_c64_mse_corr0.9_seed1
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03599  pearson 0.4559   0.5 phút
epoch  1  mse 0.01994  pearson 0.5471   1.0 phút
epoch  2  mse 0.01856  pearson 0.5705   1.5 phút
epoch  3  mse 0.01765  pearson 0.5846   2.0 phút
epoch  4  mse 0.01704  pearson 0.5940   2.5 phút
epoch  5  mse 0.01650  pearson 0.6017   3.0 phút
epoch  6  mse 0.01610  pearson 0.6086   3.5 phút
epoch  7  mse 0.01576  pearson 0.6132   4.0 phút
epoch  8  mse 0.01546  pearson 0.6196   4.5 phút
epoch  9  mse 0.01516  pearson 0.6224   4.9 phút
epoch 10  mse 0.01493  pearson 0.6252   5.4 phút
epoch 11  mse 0.01464  pearson 0.6288   5.9 phút
epoch 12  mse 0.01440  pearson 0.6311   6.4 phút
epoch 13  mse 0.01415  pearson 0.6347   6.9 phút
epoch 14  mse 0.01393  pearson 0.6362   7.4 phút
epoch 15  mse 0.01368  pearson 0.6394   7.9 phút
epoch 16  mse 0.01348

## 4. TCN đúng chuẩn Bai et al. — WeightNorm

Mục này chạy sau ba mục trên, ở một phiên Colab khác.

Bai et al. 2018 mục 3.4 dùng **WeightNorm**. Đồ án dùng **BatchNorm** cho cả hai
nhánh, lý do ghi ở `docs/THAM_CHIEU.md`: nhánh `ds_tcn` theo Howard et al. 2017
mục 3.1 vốn dùng BatchNorm, và hai nhánh phải chuẩn hoá giống nhau thì so mới cô
lập được đúng một biến là phép tích chập.

Lập luận đó chỉ đòi hai nhánh **giống nhau**, không đòi phải là BatchNorm. Nên
kết luận của mục 2 và 3 chỉ phát biểu được là *"cấu hình TCN dùng BatchNorm này
thua LSTM"*, chưa nói được gì về bản đúng chuẩn.

Mục này chạy bản đúng chuẩn để đóng lỗ hổng đó.

`ds_tcn` không cần chạy lại — nó vốn đã đúng Howard.

Hai điểm khác nhau khi bật `--norm weight`:

```
BatchNorm    chuẩn hoá ĐẦU RA:  y = (x - trung_bình_lô) / độ_lệch_lô * γ + β
             phụ thuộc lô, train và eval hành xử khác nhau

WeightNorm   viết lại TRỌNG SỐ: w = g * v / ||v||
             không phụ thuộc lô, train và eval giống hệt nhau
```

WeightNorm không phải một lớp trong luồng dữ liệu, nên khi bật thì lớp
`BatchNorm1d` bị bỏ hẳn, thay bằng `Identity`.

Kiểm bản cài đặt trước, đừng train rồi mới biết sai. Sáu phép kiểm mất vài giây;
sai một chi tiết là ba giờ train cho ra kết luận vô nghĩa.

In [4]:
# Kiểm bản cài đặt WeightNorm TRƯỚC khi train. Sai một chi tiết là ra kết luận
# "TCN đúng chuẩn cũng thua" trong khi thật ra là bản cài đặt hỏng.
import torch, torch.nn as nn
from src import models

batch = models.build_model("tcn", channels=64, norm="batch")
weight = models.build_model("tcn", channels=64, norm="weight")

def dem(m, loai):
    return sum(1 for _ in m.modules() if isinstance(_, loai))

print("1. BatchNorm bị gỡ khi bật WeightNorm")
print("   norm=batch  : %2d lớp BatchNorm1d, %2d lớp Identity"
      % (dem(batch, nn.BatchNorm1d), dem(batch, nn.Identity)))
print("   norm=weight : %2d lớp BatchNorm1d, %2d lớp Identity"
      % (dem(weight, nn.BatchNorm1d), dem(weight, nn.Identity)))
assert dem(weight, nn.BatchNorm1d) == 0, "vẫn còn BatchNorm — chưa gỡ"
assert dem(batch, nn.BatchNorm1d) == 12, "phải có 12 BatchNorm ở bản batch"

print("\n2. WeightNorm thật sự được áp lên trọng số")
co = [t for t, _ in weight.named_parameters() if "parametrizations" in t or "_g" in t]
print("   số tham số mang dấu vết WeightNorm:", len(co))
print("   ví dụ:", co[:2])
assert co, "không thấy tham số nào của WeightNorm — chưa áp được"
assert not [t for t, _ in batch.named_parameters()
            if "parametrizations" in t or "_g" in t], "bản batch không được có"

print("\n3. Số tham số")
nb, nw = models.count_params(batch), models.count_params(weight)
print("   norm=batch  : %d" % nb)
print("   norm=weight : %d   (%+d)" % (nw, nw - nb))

print("\n4. Chạy tới được, đúng shape, số hữu hạn")
x = torch.randn(4, 200)
for ten, m in [("batch", batch), ("weight", weight)]:
    m.eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (4, 25), "shape sai: %s" % (y.shape,)
    assert torch.isfinite(y).all(), "có giá trị không hữu hạn"
    print("   %-7s -> %s, hữu hạn" % (ten, tuple(y.shape)))

print("\n5. Gradient chạy được")
for ten, m in [("batch", batch), ("weight", weight)]:
    m.train()
    m(x).sum().backward()
    n_grad = sum(1 for p in m.parameters() if p.grad is not None)
    print("   %-7s -> %d tham số có gradient" % (ten, n_grad))

print("\n6. ds_tcn dùng WeightNorm bọc được CẢ HAI lớp conv")
ds = models.build_model("ds_tcn", channels=64, norm="weight")
n_conv = dem(ds, nn.Conv1d)
n_wn = len([t for t, _ in ds.named_parameters() if "parametrizations" in t or "_g" in t])
print("   %d lớp Conv1d, %d tham số WeightNorm" % (n_conv, n_wn))

print("\nTẤT CẢ ĐẠT — bản cài đặt dùng được")

1. BatchNorm bị gỡ khi bật WeightNorm
   norm=batch  : 12 lớp BatchNorm1d,  0 lớp Identity
   norm=weight :  0 lớp BatchNorm1d, 12 lớp Identity

2. WeightNorm thật sự được áp lên trọng số
   số tham số mang dấu vết WeightNorm: 24
   ví dụ: ['blocks.0.layer_one.conv.parametrizations.weight.original0', 'blocks.0.layer_one.conv.parametrizations.weight.original1']

3. Số tham số
   norm=batch  : 151513
   norm=weight : 150745   (-768)

4. Chạy tới được, đúng shape, số hữu hạn
   batch   -> (4, 25), hữu hạn
   weight  -> (4, 25), hữu hạn

5. Gradient chạy được
   batch   -> 52 tham số có gradient
   weight  -> 40 tham số có gradient

6. ds_tcn dùng WeightNorm bọc được CẢ HAI lớp conv
   25 lớp Conv1d, 48 tham số WeightNorm

TẤT CẢ ĐẠT — bản cài đặt dùng được


Sáu phép kiểm đạt thì mới chạy. **12 lần train, khoảng 1,5 giờ**.

Tên cấu hình là `tcn_c64_weight_mse_corr0.9_seed<N>` — hậu tố `_weight` chỉ xuất
hiện khi khác mặc định, nên tên của các lần chạy trước không đổi.

In [5]:
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64 --norm weight --seed 0
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64 --norm weight --seed 1
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64 --norm weight --seed 2

thực nghiệm tn1  -> runs/tn1/
cấu hình tcn_c64_weight_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03094  pearson 0.4935   0.4 phút
epoch  1  mse 0.02055  pearson 0.5571   0.9 phút
epoch  2  mse 0.01932  pearson 0.5753   1.3 phút
epoch  3  mse 0.01848  pearson 0.5881   1.8 phút
epoch  4  mse 0.01790  pearson 0.5961   2.2 phút
epoch  5  mse 0.01736  pearson 0.6026   2.7 phút
epoch  6  mse 0.01698  pearson 0.6077   3.1 phút
epoch  7  mse 0.01661  pearson 0.6115   3.6 phút
epoch  8  mse 0.01631  pearson 0.6159   4.0 phút
epoch  9  mse 0.01607  pearson 0.6184   4.5 phút
epoch 10  mse 0.01587  pearson 0.6223   5.0 phút
epoch 11  mse 0.01563  pearson 0.6248   5.4 phút
epoch 12  mse 0.01543  pearson 0.6277   5.9 phút
epoch 13  mse 0.01527  pearson 0.6300   6.3 phút
epoch 14  mse 0.01508  pearson 0.6313   6.8 phút
epoch 15  mse 0.01497  pearson 0.6335   7.2 phút
epoch 16  mse 0

## 5. Cất kết quả mục 4

`run_cv.py` đã tự nén sau mỗi fold và chép sang Drive. Ô dưới nén lại một lần
sau khi xong cả 12 lần chạy, ra tên riêng `tn1_tcn_weightnorm.zip` để không đè
tệp nén của các mục trên.

In [6]:
!python scripts/save_results.py tn1 --out tn1_tcn_weightnorm


runs/tn1/  ->  runs/tn1_tcn_weightnorm.zip   (6.9 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-06 12:44   tn1/
        0  2026-09-06 09:54   tn1/tcn_c64_weight_mse_corr0.9_seed0_val_AB/
        0  2026-09-06 10:10   tn1/tcn_c64_weight_mse_corr0.9_seed0_val_CE/
        0  2026-09-06 10:25   tn1/tcn_c64_weight_mse_corr0.9_seed0_val_DF/
        0  2026-09-06 10:40   tn1/tcn_c64_weight_mse_corr0.9_seed0_val_KL/
        0  2026-09-06 10:54   tn1/tcn_c64_weight_mse_corr0.9_seed1_val_AB/
        0  2026-09-06 11:09   tn1/tcn_c64_weight_mse_corr0.9_seed1_val_CE/
        0  2026-09-06 11:25   tn1/tcn_c64_weight_mse_corr0.9_seed1_val_DF/
        0  2026-09-06 11:40   tn1/tcn_c64_weight_mse_corr0.9_seed1_val_KL/
        0  2026-09-06 11:53   tn1/tcn_c64_weight_mse_corr0.9_seed2_val_AB/
        0  2026-09-06 12:09   tn1/tcn_c64_weight_mse_corr0.9_seed2_val_CE/
        0  2026-09-06 12:25   tn1/tcn_c64_weight_mse_corr0.9_seed2_val_DF/
        0  2026-09-06 12:39   tn1/tcn_